# Ajout des pays aux tracks

This notebook allows to add the country to the cyclone tracks.
This is useful to reduce computation time for the join between Litpop data and cyclone tracks, by filtering contry by country.

In [1]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.strtree import STRtree
from pathlib import Path
from tqdm.notebook import tqdm
import gc
import psutil
import time
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger()

def mem_mb() -> float:
    return psutil.Process().memory_info().rss / 1e6

log.info(f"Imports OK — RAM initiale : {mem_mb():.0f} MB")

20:28:00  INFO  Imports OK — RAM initiale : 179 MB


## Configuration

In [2]:
model      = "ACCESS-CM2"
experiment = "ssp245"

tracks_dir = Path(f"../data/output/catherina/intensified_tracks/{model}/{experiment}")
output_dir = Path(fr"../data/output/catherina/intensified_tracks_with_country/{model}/{experiment}")

output_dir.mkdir(parents=True, exist_ok=True)
print(f"Source : {tracks_dir}")
print(f"Sortie : {output_dir}")

Source : ../data/output/catherina/intensified_tracks/ACCESS-CM2/ssp245
Sortie : ../data/output/catherina/intensified_tracks_with_country/ACCESS-CM2/ssp245


## Chargement du shapefile pays + construction du STRtree

L'index spatial est construit **une seule fois** avant la boucle.
Utiliser la résolution 50m pour plus de précision sur les côtes :
`https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_0_countries.zip`

In [3]:
log.info("Chargement du shapefile naturalearth 110m...")
world = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
)[["NAME", "ISO_A2", "geometry"]].rename(
    columns={"NAME": "country", "ISO_A2": "country_iso2"}
)

tree          = STRtree(world.geometry)
country_names = world["country"].to_numpy()
country_iso2s = world["country_iso2"].to_numpy()

log.info(f"STRtree prêt ({len(world)} pays) — RAM : {mem_mb():.0f} MB")

20:28:12  INFO  Chargement du shapefile naturalearth 110m...


ERROR 1: PROJ: proj_create_from_database: Open of /home/tiphanie/projets_python/catherina/.pixi/envs/default/share/proj failed


20:28:15  INFO  STRtree prêt (177 pays) — RAM : 209 MB


## Fonction d'enrichissement

**Optimisation clé :** on ne fait la recherche spatiale que sur les points `is_on_land=True`.
Les points en mer (la grande majorité des tracks) reçoivent directement `NaN` sans aucun calcul.

In [5]:
def add_country_fast(
    df: pd.DataFrame,
    tree: STRtree,
    country_names: np.ndarray,
    country_iso2s: np.ndarray,
) -> pd.DataFrame:
    """
    Ajoute les colonnes 'country' et 'country_iso2'.
    Seuls les points is_on_land=True sont soumis à la requête spatiale.
    """
    n = len(df)
    country_out  = np.full(n, None, dtype=object)
    country_iso2 = np.full(n, None, dtype=object)

    land_mask = df["is_on_land"].to_numpy(dtype=bool)
    n_land = land_mask.sum()

    if n_land > 0:
        land_positions = np.where(land_mask)[0]
        lons = df["lon_left"].to_numpy()[land_mask]
        lats = df["lat_left"].to_numpy()[land_mask]

        # Création des points shapely (vectorisé)
        land_points = gpd.points_from_xy(lons, lats)

        # Requête vectorisée : point_idxs et geom_idxs sont des arrays parallèles
        point_idxs, geom_idxs = tree.query(land_points, predicate="within")

        # En cas de doublons (point sur frontière commune), on garde le premier match
        seen = set()
        for pt_i, geo_i in zip(point_idxs, geom_idxs):
            if pt_i not in seen:
                seen.add(pt_i)
                df_row = land_positions[pt_i]
                country_out[df_row]  = country_names[geo_i]
                country_iso2[df_row] = country_iso2s[geo_i]

    out = df.copy()
    out["country"]      = country_out
    out["country_iso2"] = country_iso2
    return out

## Détection des chunks (seed × année) à traiter

On construit la liste des paires `(seed, year)` à partir des répertoires existants,
et on exclut celles déjà présentes en sortie (reprise automatique).

In [6]:
# Paires (seed, year) disponibles en entrée — lecture depuis l'arborescence de fichiers
# (pas de dataset pyarrow global, pour ne pas charger tous les fichiers en mémoire)
input_chunks = sorted(
    (int(seed_dir.name.split("=")[1]), int(year_dir.name.split("=")[1]))
    for seed_dir in sorted(tracks_dir.glob("seed=*"))
    for year_dir in sorted(seed_dir.glob("year=*"))
)

# Paires déjà traitées en sortie
done_chunks = set(
    (int(seed_dir.name.split("=")[1]), int(year_dir.name.split("=")[1]))
    for seed_dir in sorted(output_dir.glob("seed=*"))
    for year_dir in sorted(seed_dir.glob("year=*"))
) if output_dir.exists() else set()

chunks_to_process = [c for c in input_chunks if c not in done_chunks]

all_seeds = sorted({s for s, _ in input_chunks})
log.info(f"Seeds disponibles     : {len(all_seeds)}")
log.info(f"Chunks total          : {len(input_chunks)}")
log.info(f"Chunks déjà traités   : {len(done_chunks)}")
log.info(f"Chunks restants       : {len(chunks_to_process)}")

20:28:25  INFO  Seeds disponibles     : 1
20:28:25  INFO  Chunks total          : 75
20:28:25  INFO  Chunks déjà traités   : 0
20:28:25  INFO  Chunks restants       : 75


## Traitement par chunk (seed × année)

Chaque itération lit **un seul répertoire** `seed=X/year=Y/` directement via `pq.read_table`
(pas de `dataset.filter()` global), enrichit les points terrestres, et écrit immédiatement.
La RAM utilisée par chunk est ~1–5 MB au lieu de ~128–322 MB.

Les logs affichent la RAM à chaque étape pour identifier où un crash éventuel se produirait.

In [7]:
t_global = time.time()
errors = []

for chunk_idx, (seed, year) in enumerate(
    tqdm(chunks_to_process, desc="Chunks")
):

    t0 = time.time()
    chunk_label = f"seed={seed}/year={year}"
    in_dir = tracks_dir / f"seed={seed}" / f"year={year}"
    out_dir = output_dir / f"seed={seed}" / f"year={year}"

    log.info(
        f"[{chunk_idx+1}/{len(chunks_to_process)}] {chunk_label} — "
        f"RAM avant lecture : {mem_mb():.0f} MB"
    )

    table_chunk = pq.read_table(in_dir)

    log.info(
        f"  Lecture OK : {table_chunk.num_rows:,} lignes — "
        f"RAM : {mem_mb():.0f} MB"
    )

    df_chunk = table_chunk.to_pandas()

    del table_chunk
    gc.collect()

    # Si seed/year/month sont dans l'index pandas (metadata parquet),
    # on les ramène en colonnes normales
    if isinstance(df_chunk.index, pd.MultiIndex) or df_chunk.index.name is not None:
        df_chunk = df_chunk.reset_index()

    log.info(
        f"  to_pandas OK : colonnes={list(df_chunk.columns)[:6]}... — "
        f"RAM : {mem_mb():.0f} MB"
    )

    # ── 2. Filtrage des points sur terre ──────────────────────────────────
    n_before = len(df_chunk)

    mask = df_chunk["is_on_land"]
    n_land = int(mask.sum())

    log.info(
        f"  Points sur terre : {n_land:,} / {n_before:,}"
    )

    df_chunk = df_chunk.loc[mask]

    gc.collect()

    log.info(
        f"  Filtrage OK : {len(df_chunk):,} lignes conservées — "
        f"RAM : {mem_mb():.0f} MB"
    )

    # ── 3. Enrichissement pays ────────────────────────────────────────────
    df_enriched = add_country_fast(
        df_chunk,
        tree,
        country_names,
        country_iso2s,
    )

    del df_chunk
    gc.collect()

    log.info(
        f"  Enrichissement OK — RAM : {mem_mb():.0f} MB"
    )

    # ── 4. Écriture par sous-répertoire month= ────────────────────────────
    for month, df_month in df_enriched.groupby("month"):
        month_out = out_dir / f"month={month}"
        month_out.mkdir(parents=True, exist_ok=True)

        pq.write_table(
            pa.Table.from_pandas(
                df_month,
                preserve_index=False,
            ),
            month_out / "part-0.parquet",
        )

    del df_enriched
    gc.collect()

    elapsed = time.time() - t0

    log.info(
        f"  Écriture OK — {elapsed:.1f}s — "
        f"RAM : {mem_mb():.0f} MB"
    )

total_elapsed = time.time() - t_global

log.info(
    f"=== Terminé en {total_elapsed/60:.1f} min — "
    f"{len(chunks_to_process) - len(errors)} OK, "
    f"{len(errors)} erreurs ==="
)

if errors:
    log.warning("Chunks en erreur :")
    for label, msg in errors:
        log.warning(f"  {label}: {msg}")

Chunks:   0%|          | 0/75 [00:00<?, ?it/s]

20:28:30  INFO  [1/75] seed=0/year=2025 — RAM avant lecture : 212 MB
20:28:30  INFO    Lecture OK : 1,960 lignes — RAM : 233 MB
20:28:30  INFO    to_pandas OK : colonnes=['SID', 'step', 'index', 'lat_left', 'lon_left', 'datetime']... — RAM : 235 MB
20:28:30  INFO    Points sur terre : 198 / 1,960
20:28:30  INFO    Filtrage OK : 198 lignes conservées — RAM : 235 MB
20:28:30  INFO    Enrichissement OK — RAM : 235 MB
20:28:30  INFO    Écriture OK — 0.4s — RAM : 236 MB
20:28:30  INFO  [2/75] seed=0/year=2026 — RAM avant lecture : 236 MB
20:28:30  INFO    Lecture OK : 2,313 lignes — RAM : 241 MB


/tmp/ipykernel_21901/4185353495.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for month, df_month in df_enriched.groupby("month"):


20:28:30  INFO    to_pandas OK : colonnes=['SID', 'step', 'index', 'lat_left', 'lon_left', 'datetime']... — RAM : 243 MB
20:28:30  INFO    Points sur terre : 105 / 2,313
20:28:30  INFO    Filtrage OK : 105 lignes conservées — RAM : 243 MB
20:28:30  INFO    Enrichissement OK — RAM : 243 MB
20:28:30  INFO    Écriture OK — 0.4s — RAM : 243 MB
20:28:30  INFO  [3/75] seed=0/year=2027 — RAM avant lecture : 243 MB


/tmp/ipykernel_21901/4185353495.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for month, df_month in df_enriched.groupby("month"):


...

/tmp/ipykernel_21901/4185353495.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for month, df_month in df_enriched.groupby("month"):


20:28:58  INFO    to_pandas OK : colonnes=['SID', 'step', 'index', 'lat_left', 'lon_left', 'datetime']... — RAM : 305 MB
20:28:58  INFO    Points sur terre : 269 / 4,032
20:28:58  INFO    Filtrage OK : 269 lignes conservées — RAM : 305 MB
20:28:58  INFO    Enrichissement OK — RAM : 305 MB
20:28:58  INFO    Écriture OK — 0.4s — RAM : 305 MB
20:28:58  INFO  [75/75] seed=0/year=2099 — RAM avant lecture : 305 MB
20:28:58  INFO    Lecture OK : 2,320 lignes — RAM : 306 MB


/tmp/ipykernel_21901/4185353495.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for month, df_month in df_enriched.groupby("month"):


20:28:58  INFO    to_pandas OK : colonnes=['SID', 'step', 'index', 'lat_left', 'lon_left', 'datetime']... — RAM : 306 MB
20:28:58  INFO    Points sur terre : 102 / 2,320
20:28:58  INFO    Filtrage OK : 102 lignes conservées — RAM : 306 MB
20:28:58  INFO    Enrichissement OK — RAM : 306 MB
20:28:59  INFO    Écriture OK — 0.4s — RAM : 306 MB
20:28:59  INFO  === Terminé en 0.5 min — 75 OK, 0 erreurs ===


/tmp/ipykernel_21901/4185353495.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for month, df_month in df_enriched.groupby("month"):


## Vérification du résultat

In [10]:
out_seeds = sorted(
    {int(p.name.split("=")[1]) for p in output_dir.glob("seed=*") if p.is_dir()}
)
print(f"Seeds en entrée : {len(all_seeds)}")
print(f"Seeds en sortie : {len(out_seeds)}")

missing = sorted(set(all_seeds) - set(out_seeds))
if missing:
    print(f"Seeds manquants : {missing}")
else:
    print("Tous les seeds sont présents en sortie.")

Seeds en entrée : 1
Seeds en sortie : 1
Tous les seeds sont présents en sortie.


In [11]:
result_ds = ds.dataset(output_dir, partitioning="hive")
sample = result_ds.head(2000).to_pandas()
print(f"Colonnes : {list(sample.columns)}")
print()
print("Top pays :")
print(sample["country"].value_counts().head(10))
print()
sample[["lat_left", "lon_left", "is_on_land", "country", "country_iso2", "seed", "year"]].head(10)

Colonnes : ['SID', 'step', 'index', 'lat_left', 'lon_left', 'datetime', 'time', 'y', 'x', 'nshr', 'mslp_hpa', 'T_strat', 'SST', 'height', 'lat_right', 'lon_right', 'distance_track_env', 'thermo_eff', 'basin', 'A', 'B', 'C', 'a', 'b', 'c_0', 'c_1', 'c_2', 'c_3', 'sigma_pc', 'q_env', 'wind_speed', 'delta_pc_hpa', 'pc_hpa', 'eps_pc', 'is_on_land', 'dist2coast_meters', 'V_0', 't_L_steps', 'final_wind_speed', 'delta_pc_hpa_1', 'pc_hpa_1', 'final_wind_1', 'final_wind_2', 'V_0_1', 't_L_steps_1', 'pc_hpa_1_eff', 'qc', 'delta_Sm', 'X', 'MPI_hpa', 'mpd_hpa', 'mslp_mpd_diff_hpa', 'month', 'country', 'country_iso2', 'seed', 'year']

Top pays :
country
India                       108
Pakistan                     18
Philippines                  17
Nicaragua                    13
United States of America      9
China                         9
Mexico                        5
Papua New Guinea              4
Canada                        3
Nepal                         3
Name: count, dtype: int64



,lat_left,lon_left,is_on_land,country,country_iso2,seed,year
0,12.500484,-106.500000,False,None,None,0,2025
1,12.461908,-106.572353,False,None,None,0,2025
2,12.434372,-106.662559,False,None,None,0,2025
3,12.417464,-106.769879,False,None,None,0,2025
4,12.410696,-106.893604,False,None,None,0,2025
5,12.413507,-107.033055,False,None,None,0,2025
6,12.425279,-107.187582,False,None,None,0,2025
7,12.445342,-107.356561,False,None,None,0,2025
8,12.472985,-107.539393,False,None,None,0,2025
9,12.501621,-107.838640,False,None,None,0,2025
